# CSPF — second annotator: agreement (κ) and re-computed human-audit statistics
**Input:** (1) your annotation file (`annotated_output.csv`, columns `meaning_preserving`, `sentiment_preserving`, `natural`), (2) the sheet filled by the second annotator (`annotation_sheet_for_second_annotator.csv`, columns `sample_id, meaning_a2, sentiment_a2, natural_a2`).
**Output:** Cohen's κ (unweighted and linear-weighted) with bootstrap CI, raw agreement, confusion matrices, per-family κ, a list of disagreements for adjudication, and the headline numbers recomputed for annotator 1, annotator 2 and the strict consensus. CPU only, runs in seconds.
**Who should annotate?** A person who did NOT see annotator 1's labels and is fluent in English (a colleague, a PhD student). Do not use Claude or another LLM as 'annotator 2' for the paper; an LLM judge can be reported separately as an automatic judge.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
A1_CSV = "/content/drive/MyDrive/postdoc-papers/paper2/annotated_output.csv"
A2_CSV = "/content/drive/MyDrive/postdoc-papers/paper2/annotation_sheet_for_second_annotator_FILLED.csv"
OUT_DIR = "/content/drive/MyDrive/postdoc-papers/paper2/results_FIXED/annotation"
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score, roc_auc_score, confusion_matrix
a1 = pd.read_csv(A1_CSV, encoding="utf-8-sig"); a2 = pd.read_csv(A2_CSV, encoding="utf-8-sig")
for c in ("meaning_a2","sentiment_a2","natural_a2"): a2[c] = a2[c].astype(str).str.strip().str.lower()
ORD = {"meaning":("meaning_preserving","meaning_a2",["no","partial","yes"]),
       "sentiment":("sentiment_preserving","sentiment_a2",["no","unclear","yes"]),
       "natural":("natural","natural_a2",["no","borderline","yes"])}
d = a1.merge(a2[["sample_id","meaning_a2","sentiment_a2","natural_a2"]], on="sample_id", how="inner")
print(len(a1), "rows in A1;", len(a2), "in A2;", len(d), "matched")
bad = {k:sorted(set(d[c2])-set(lv)) for k,(c1,c2,lv) in ORD.items()}
assert all(len(v)==0 for v in bad.values()), f"unexpected labels in annotator 2 file: {bad} (blank or misspelled?)"

In [ ]:
rng = np.random.default_rng(0)
def boot_kappa(x, y, levels, weights, B=2000):
    n=len(x); ks=[]
    for _ in range(B):
        i=rng.integers(0,n,n)
        try: ks.append(cohen_kappa_score(x[i], y[i], labels=levels, weights=weights))
        except Exception: pass
    return np.nanpercentile(ks,[2.5,97.5])
rows=[]
for k,(c1,c2,lv) in ORD.items():
    x=d[c1].values; y=d[c2].values
    for w in (None,"linear"):
        kap=cohen_kappa_score(x,y,labels=lv,weights=w); lo,hi=boot_kappa(x,y,lv,w)
        rows.append(dict(dimension=k,weighting=w or "none",kappa=round(kap,3),ci_lo=round(lo,3),ci_hi=round(hi,3),agreement=round((x==y).mean(),3)))
K=pd.DataFrame(rows); print(K.to_string(index=False)); K.to_csv(f"{OUT_DIR}/kappa_overall.csv",index=False)
print("\nLandis & Koch: <0.20 slight, .21-.40 fair, .41-.60 moderate, .61-.80 substantial, >.80 almost perfect")
for k,(c1,c2,lv) in ORD.items():
    cm=pd.DataFrame(confusion_matrix(d[c1],d[c2],labels=lv),index=[f"A1_{l}" for l in lv],columns=[f"A2_{l}" for l in lv]); print("\n",k); print(cm)
    cm.to_csv(f"{OUT_DIR}/confusion_{k}.csv")

In [ ]:
# per-family kappa on the meaning judgement (48 pairs per family -> wide uncertainty, report as descriptive)
c1,c2,lv = ORD["meaning"]; r=[]
for fam,g in d.groupby("perturbation_family"):
    x,y=g[c1].values,g[c2].values
    kap = cohen_kappa_score(x,y,labels=lv,weights="linear") if len(set(x))>1 or len(set(y))>1 else np.nan
    r.append(dict(family=fam,n=len(g),agreement=round((x==y).mean(),3),kappa_linear=None if np.isnan(kap) else round(kap,3)))
F=pd.DataFrame(r); print(F.to_string(index=False)); F.to_csv(f"{OUT_DIR}/kappa_by_family.csv",index=False)
# disagreements -> adjudicate together, record the final label in a NEW column; never overwrite A1/A2
dis = d[(d.meaning_preserving!=d.meaning_a2)|(d.sentiment_preserving!=d.sentiment_a2)|(d.natural!=d.natural_a2)]
dis[["sample_id","perturbation_family","original_review","perturbed_review","meaning_preserving","meaning_a2","sentiment_preserving","sentiment_a2","natural","natural_a2"]].assign(adjudicated_meaning="",adjudicated_sentiment="",adjudicated_natural="").to_csv(f"{OUT_DIR}/disagreements_to_adjudicate.csv",index=False)
print(len(dis),"rows with at least one disagreement ->", f"{OUT_DIR}/disagreements_to_adjudicate.csv")

In [ ]:
# headline numbers of the paper, recomputed per annotator and for the strict consensus (both agree)
PROP=["P02_discourse","P08_synonym","P10_syntax","P11_entity","P12_register"]; MOD=["P03_negation","P04_sarcasm","P05_dilution","P07_contrastive","P09_intensity"]
def summ(lab, name):
    s=lab.copy(); out=dict(annotator=name, n=len(s))
    out["meaning_yes_partial_no"]="/".join(str(int((s.m==v).sum())) for v in ("yes","partial","no"))
    out["share_fully_preserving_propositional"]=round((s[s.perturbation_family.isin(PROP)].m=="yes").mean(),3)
    out["share_fully_preserving_modulating"]=round((s[s.perturbation_family.isin(MOD)].m=="yes").mean(),3)
    out["AUC_phi_vs_preserving"]=round(roc_auc_score(s.m=="yes", s.sem_preservation_score),3)
    out["AUC_phi_vs_not_altering"]=round(roc_auc_score(s.m!="no", s.sem_preservation_score),3) if (s.m=="no").any() else None
    return out
t=[summ(d.assign(m=d.meaning_preserving),"annotator 1"), summ(d.assign(m=d.meaning_a2),"annotator 2")]
cons=d[d.meaning_preserving==d.meaning_a2].assign(m=lambda x:x.meaning_preserving); t.append(summ(cons,"consensus (both agree)"))
T=pd.DataFrame(t); print(T.to_string(index=False)); T.to_csv(f"{OUT_DIR}/humanaudit_headlines.csv",index=False)